# Process v3 stop-level delay data for the dashboard

Same inputs and outputs as `process_delay_data_v2.ipynb`, with one
addition: each route in `network_routes.json` now carries a `path`
field — a list of `[lat, lon]` vertices from the v2 aggregator's
routed LineString (OSM-snapped where available, fallback otherwise).

Why: the v2 dashboard drew each route as straight stop-to-stop
polylines, which short-cuts curves and wrong-side dual carriageways.
The aggregator already stores a properly routed LineString per
`(route_id, direction_id)` in the `routes` layer of
`stop_metrics_v2.gpkg`; this notebook downsamples it (Douglas–Peucker
via `shapely.simplify`) and serialises it alongside the existing
stop list. Old consumers that only read `stops` keep working; new
consumers can prefer `path` for the rendered polyline.

## What changed since v2

* **`path` per route** — `[[lat, lon], …]` along the canonical
  direction's LineString, simplified to `PATH_SIMPLIFY_TOL` degrees
  and rounded to `PATH_COORD_DECIMALS` decimal places. Tune both to
  trade fidelity vs. payload size.
* **No schema break** — `route_details.json` is identical to v2;
  `network_routes.json` is a strict superset.

Defaults (`tol=1e-5°`, `decimals=5`) keep ~1 m of detail on the
ground while typically only ~doubling the network JSON size.

In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString

# ── Inputs ─────────────────────────────────────────────────────────
PROJ_DIR  = Path(r"D:\2026_03-Bus_Project")
INPUT_DIR = PROJ_DIR / "output" / "aggregate"
STOP_METRICS_GPKG = INPUT_DIR / "stop_metrics_v2.gpkg"
WEEKDAY_CSV       = INPUT_DIR / "stop_metrics_weekday_v2.csv"
WEEKEND_CSV       = INPUT_DIR / "stop_metrics_weekend_v2.csv"

# ── Output ─────────────────────────────────────────────────────────
OUTPUT_DIR = Path('.').resolve()

# ── Filters (unchanged from v2) ──────────────────────────────────────
N_ROUTES = None
ALLOWED_ROUTE_TYPES = ["Bus"]
MIN_STOPS_PER_ROUTE = 4
# Drop routes whose total weekday `n_arrivals_day` (summed across all stops)
# is below this threshold — keeps the dashboard focused on routes with
# enough observations to be statistically meaningful. Set to None to disable.
MIN_ARRIVALS_PER_ROUTE = 5000
MAX_TRIPS_PER_PERIOD = 12

# ── NEW: path simplification ──────────────────────────────────────────
# Douglas–Peucker tolerance, in degrees (CRS is EPSG:4326). Roughly:
#   1e-4 ≈ 11 m, 5e-5 ≈ 5 m, 1e-5 ≈ 1 m.
PATH_SIMPLIFY_TOL = 1e-5
# Coordinate rounding for the JSON payload.
PATH_COORD_DECIMALS = 5

# ── Period mapping (short ↔ long names from the v2 aggregator) ──────────
PERIOD_COLS = {
    'all':     'day',
    'am_off':  'morning_offpeak',
    'am_peak': 'morning_peak',
    'midday':  'midday_offpeak',
    'pm_peak': 'evening_peak',
    'pm_off':  'evening_offpeak',
}
DETAIL_PERIODS = [k for k in PERIOD_COLS if k != 'all']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"input  : {INPUT_DIR}")
print(f"output : {OUTPUT_DIR}")
print(f"filters: route_type_name={ALLOWED_ROUTE_TYPES}, "
      f"N_ROUTES={'all' if N_ROUTES is None else N_ROUTES}, "
      f"min_arrivals={MIN_ARRIVALS_PER_ROUTE}")
print(f"path   : simplify_tol={PATH_SIMPLIFY_TOL} deg, decimals={PATH_COORD_DECIMALS}")

input  : D:\2026_03-Bus_Project\output\aggregate
output : C:\Users\Lenovo\Desktop\UCL_Moodle\MT2\CASA0029-Urban_Data_Visualisation\Assignments\Group_Visualisation\CASA0029-Group-18-Bus-Delay-in-Manchester\data\delay
filters: route_type_name=['Bus'], N_ROUTES=all, min_arrivals=5000
path   : simplify_tol=1e-05 deg, decimals=5


## 1. Load and filter stop-metrics CSVs

Identical to v2.

In [2]:
def load_metrics(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path,
                     dtype={"route_id": str, "direction_id": str, "stop_id": str},
                     low_memory=False)
    if ALLOWED_ROUTE_TYPES is not None:
        before = len(df)
        df = df[df["route_type_name"].isin(ALLOWED_ROUTE_TYPES)]
        print(f"  {path.name}: {before:,} → {len(df):,} rows after type filter")
    else:
        print(f"  {path.name}: {len(df):,} rows (no type filter)")
    df["stop_name"] = df["stop_name"].fillna("")
    return df.reset_index(drop=True)


df_wd = load_metrics(WEEKDAY_CSV)
df_we = load_metrics(WEEKEND_CSV)
if df_wd.empty:
    raise RuntimeError(
        f"No weekday rows survived the route_type_name filter "
        f"{ALLOWED_ROUTE_TYPES}. Check ALLOWED_ROUTE_TYPES.")

print(f"\nweekday: {len(df_wd):,} rows · weekend: {len(df_we):,} rows")
print(f"unique route_short_name (weekday): {df_wd['route_short_name'].nunique()}")

  stop_metrics_weekday_v2.csv: 44,935 → 44,935 rows after type filter
  stop_metrics_weekend_v2.csv: 22,513 → 22,513 rows after type filter

weekday: 44,935 rows · weekend: 22,513 rows
unique route_short_name (weekday): 574


## 2. Load route geometries and ordered stop sequences

Same as v2 but we **also keep the LineString geometry** of the canonical
direction so we can emit a high-fidelity `path` per route in step 4. The
`primary` table now indexes both `stop_sequence` and `geometry` by
`route_short_name`.

In [3]:
gdf_routes = gpd.read_file(STOP_METRICS_GPKG, layer="routes")

if ALLOWED_ROUTE_TYPES is not None:
    before = len(gdf_routes)
    gdf_routes = gdf_routes[gdf_routes["route_type_name"].isin(ALLOWED_ROUTE_TYPES)]
    print(f"  routes layer: {before:,} → {len(gdf_routes):,} rows after type filter")
else:
    print(f"  routes layer: {len(gdf_routes):,} rows")

if gdf_routes.crs is not None and gdf_routes.crs.to_epsg() != 4326:
    gdf_routes = gdf_routes.to_crs(4326)

gdf_routes["stop_sequence"] = gdf_routes["stop_sequence"].apply(
    lambda s: json.loads(s) if isinstance(s, str)
    else (list(s) if s is not None else [])
)

primary = (gdf_routes.sort_values("n_stops_in_area", ascending=False)
                     .drop_duplicates("route_short_name", keep="first")
                     .set_index("route_short_name"))

print(f"\n{len(primary):,} short_names have a canonical stop_sequence + geometry")
print(primary[["route_id", "direction_id", "n_stops_in_area", "geometry_method"]].head().to_string())

  routes layer: 1,305 → 1,179 rows after type filter

591 short_names have a canonical stop_sequence + geometry
                 route_id direction_id  n_stops_in_area geometry_method
route_short_name                                                       
41                  92514            0              118           mixed
356                  9947            1              114           mixed
35                  81812            1              113           mixed
343                 94552            1              108           mixed
149                 92592            0              106           mixed


## 3. Rank routes and pick the top N

Identical to v2.

In [4]:
route_volume = (df_wd.groupby('route_short_name')['n_arrivals_day']
                       .sum().sort_values(ascending=False))

n_total = len(route_volume)
if MIN_ARRIVALS_PER_ROUTE is not None:
    route_volume = route_volume[route_volume >= MIN_ARRIVALS_PER_ROUTE]
    print(f"min_arrivals filter: {n_total:,} → {len(route_volume):,} routes "
          f"(threshold {MIN_ARRIVALS_PER_ROUTE:,} weekday arrivals)")

if N_ROUTES is None:
    selected = route_volume.index.tolist()
    print(f"all {len(selected):,} routes selected (no N_ROUTES limit)")
else:
    selected = route_volume.head(N_ROUTES).index.tolist()
    print(f"top {len(selected)} of {len(route_volume)} routes selected by weekday arrivals")

print(f"first 10: {selected[:10]}")

min_arrivals filter: 574 → 183 routes (threshold 5,000 weekday arrivals)
all 183 routes selected (no N_ROUTES limit)
first 10: ['192', '36', '163', '17', '84', '135', '471', '201', '409', '52']


## 4. Build `network_routes.json` (now with `path`)

For each selected route we still emit `name`, `stops`, and the per-period
weighted aggregates from v2. **New:** a `path` field carrying the
simplified routed LineString as `[[lat, lon], …]` so the dashboard can
draw the actual road geometry instead of straight stop-to-stop lines.

MultiLineStrings (rare; from disjoint route shapes) are flattened by
concatenating their parts in storage order.

In [5]:
def weighted_period_stats(stops_df: pd.DataFrame, period_col: str) -> dict:
    n_col   = f'n_arrivals_{period_col}'
    q2_col  = f'Q2_{period_col}'
    otp_col = f'pct_on_time_{period_col}'
    s = stops_df.dropna(subset=[q2_col])
    if s.empty or s[n_col].sum() == 0:
        return {'mean': 0.0, 'otp': 0.0, 'n': 0}
    w = s[n_col].astype(float)
    return {
        'mean': float((s[q2_col] * w).sum() / w.sum()),
        'otp':  float((s[otp_col].fillna(0) * w).sum() / w.sum()),
        'n':    int(w.sum()),
    }


def busiest_by_stop(df: pd.DataFrame) -> pd.DataFrame:
    return (df.sort_values('n_arrivals_day', ascending=False)
              .drop_duplicates(['route_short_name', 'stop_id']))


def geom_to_path(geom, tol=PATH_SIMPLIFY_TOL, decimals=PATH_COORD_DECIMALS):
    '''Simplify a (Multi)LineString and return [[lat, lon], …].'''
    if geom is None or geom.is_empty:
        return []
    if tol and tol > 0:
        geom = geom.simplify(tol, preserve_topology=False)
    if isinstance(geom, LineString):
        coords = list(geom.coords)
    elif isinstance(geom, MultiLineString):
        coords = []
        for part in geom.geoms:
            coords.extend(list(part.coords))
    else:
        return []
    return [[round(float(y), decimals), round(float(x), decimals)] for x, y in coords]


df_wd_busy = busiest_by_stop(df_wd)

network = []
skipped_no_seq  = 0
skipped_too_few = 0
path_vertex_total = 0

for short_name in selected:
    if short_name not in primary.index:
        skipped_no_seq += 1
        continue
    seq = primary.at[short_name, "stop_sequence"]
    if not isinstance(seq, list) or len(seq) < MIN_STOPS_PER_ROUTE:
        skipped_too_few += 1
        continue

    sub = df_wd_busy[df_wd_busy['route_short_name'] == short_name].set_index('stop_id')
    stops_out = []
    for sid in seq:
        if sid not in sub.index:
            continue
        r = sub.loc[sid]
        stops_out.append({
            'id':   sid,
            'name': r['stop_name'],
            'lat':  round(float(r['lat']), 5),
            'lon':  round(float(r['lon']), 5),
        })
    if len(stops_out) < MIN_STOPS_PER_ROUTE:
        skipped_too_few += 1
        continue

    path = geom_to_path(primary.at[short_name, "geometry"])
    path_vertex_total += len(path)

    rows_out = sub.loc[[s['id'] for s in stops_out]]
    network.append({
        'name':    str(short_name),
        'stops':   stops_out,
        'path':    path,
        'periods': {k: weighted_period_stats(rows_out, col)
                    for k, col in PERIOD_COLS.items()},
    })

print(f"network: {len(network)} routes "
      f"(skipped: {skipped_no_seq} no-sequence, {skipped_too_few} too-few-stops)")
if network:
    avg_stops = sum(len(r['stops']) for r in network) / len(network)
    avg_path  = path_vertex_total / len(network)
    print(f"mean stops per route: {avg_stops:.1f}")
    print(f"mean path vertices:   {avg_path:.1f}")
    print(f"total path vertices:  {path_vertex_total:,}")

network: 183 routes (skipped: 0 no-sequence, 0 too-few-stops)
mean stops per route: 56.6
mean path vertices:   213.9
total path vertices:  39,145


## 5. Build `route_details.json`

Identical to v2 — the detail file is unchanged.

In [6]:
def parse_delay_list(raw):
    if pd.isna(raw):
        return []
    try:
        return [round(float(x), 2) for x in ast.literal_eval(raw)]
    except (ValueError, SyntaxError):
        return []


def even_sample(arr, k):
    if len(arr) <= k:
        return arr
    idx = np.linspace(0, len(arr) - 1, k).round().astype(int)
    return [arr[i] for i in idx]


def stop_period_record(row, period_col):
    n = row.get(f'n_arrivals_{period_col}', 0)
    if pd.isna(n) or n == 0:
        return {'mean': None, 'otp': None, 'n': 0, 'd': []}
    delays = even_sample(parse_delay_list(row.get(f'list_delay_{period_col}')),
                         MAX_TRIPS_PER_PERIOD)
    q2  = row[f'Q2_{period_col}']
    otp = row[f'pct_on_time_{period_col}']
    return {
        'mean': None if pd.isna(q2)  else round(float(q2),  2),
        'otp':  None if pd.isna(otp) else round(float(otp), 1),
        'n':    int(n),
        'd':    delays,
    }


def empty_period_block():
    return {p: {'mean': None, 'otp': None, 'n': 0, 'd': []} for p in DETAIL_PERIODS}


df_we_busy = busiest_by_stop(df_we)

details = {}
for r in network:
    sn = r['name']
    rows_wd = df_wd_busy[df_wd_busy['route_short_name'] == sn].set_index('stop_id')
    rows_we = df_we_busy[df_we_busy['route_short_name'] == sn].set_index('stop_id')
    by_stop = {}
    for stop in r['stops']:
        sid = stop['id']
        wd_row = rows_wd.loc[sid] if sid in rows_wd.index else None
        we_row = rows_we.loc[sid] if sid in rows_we.index else None
        by_stop[sid] = {
            'wd': ({p: stop_period_record(wd_row, PERIOD_COLS[p]) for p in DETAIL_PERIODS}
                   if wd_row is not None else empty_period_block()),
            'we': ({p: stop_period_record(we_row, PERIOD_COLS[p]) for p in DETAIL_PERIODS}
                   if we_row is not None else empty_period_block()),
        }
    details[sn] = by_stop

print(f"detail records for {len(details)} routes")

detail records for 183 routes


## 6. Write the JSON files

In [7]:
summary_path = OUTPUT_DIR / 'network_routes.json'
details_path = OUTPUT_DIR / 'route_details.json'

with summary_path.open('w', encoding='utf-8') as f:
    json.dump({'routes': network}, f, separators=(',', ':'))
with details_path.open('w', encoding='utf-8') as f:
    json.dump(details, f, separators=(',', ':'))

for p in (summary_path, details_path):
    print(f'{p.name}: {p.stat().st_size / 1024:.1f} KB  ->  {p}')

network_routes.json: 1598.4 KB  ->  C:\Users\Lenovo\Desktop\UCL_Moodle\MT2\CASA0029-Urban_Data_Visualisation\Assignments\Group_Visualisation\CASA0029-Group-18-Bus-Delay-in-Manchester\data\delay\network_routes.json
route_details.json: 9446.6 KB  ->  C:\Users\Lenovo\Desktop\UCL_Moodle\MT2\CASA0029-Urban_Data_Visualisation\Assignments\Group_Visualisation\CASA0029-Group-18-Bus-Delay-in-Manchester\data\delay\route_details.json


In [8]:
# Quick sanity check on one route.
if network:
    r = network[0]
    print(f"route {r['name']} · {len(r['stops'])} stops · {len(r['path'])} path vertices")
    for k, v in r['periods'].items():
        m = v['mean'] if v['mean'] is not None else float('nan')
        o = v['otp']  if v['otp']  is not None else float('nan')
        print(f"  {k:8s} mean={m:7.2f} min  otp={o:5.1f}%  n={v['n']}")
    if r['path']:
        print(f"\nfirst 3 path verts (lat, lon): {r['path'][:3]}")
        print(f"last 3:                        {r['path'][-3:]}")
else:
    print("(no routes — check filters)")

route 192 · 57 stops · 181 path vertices
  all      mean=   0.82 min  otp= 28.3%  n=78903
  am_off   mean=  12.72 min  otp= 32.3%  n=8694
  am_peak  mean=  -0.13 min  otp= 32.8%  n=8713
  midday   mean=   0.33 min  otp= 28.5%  n=30253
  pm_peak  mean=   1.89 min  otp= 20.0%  n=15253
  pm_off   mean=   1.22 min  otp= 30.9%  n=15990

first 3 path verts (lat, lon): [[53.37369, -2.11325], [53.37511, -2.11366], [53.37558, -2.11388]]
last 3:                        [[53.47942, -2.23507], [53.47977, -2.23553], [53.48066, -2.23511]]
